# MODEL_BASELINE — NextWatch

Comparaison **dummy / baseline / 3 modeles**.

| Nom | Type | Idee |
| --- | --- | --- |
| **Dummy** | — | Films au hasard (plancher). |
| **Baseline** | heuristique | Meme genres + note moyenne. |
| **User–User CF** | **non supervise** | Voisins = fans des films d'entrée proches en **age/sexe**. |
| **Film–Film CF (TF-IDF)** | **non supervise** | Voisins films (co-notes pondérées TF-IDF sur 1 à 3 films). |
| **Hybride Film×User** | **non supervise** | Film–Film (TF-IDF + genre boost) enrichi par User–User (age/sexe). |

**Metriques :** Precision@5 / @10 (aime = note >= 4). **Seed :** 42.
Entrée produit = **age + sexe + 1 à 3 films aimés**.


## 1. Chargement et verification

Meme perimetre que l'EDA : exclusion `occupation` / `unknown`, genres en liste.


In [125]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import NearestNeighbors

# --- Hyperparametres optimises (Optuna & Option B : 1 à 3 films) ---
RANDOM_STATE = 42
LIKED_THRESHOLD = 4
MAX_SEEDS = 3  # Entre 1 et 3 films d'entrée en situation produit

TRAIN_FRAC = 0.25
MIN_SUPPORT = 19
CF_N_USER = 310   # voisins User–User (fans)
CF_N_FILM = 80    # voisins Film–Film (items)
HYBRID_ALPHA = 0.05
HYBRID_POOL = 250
DEMO_SEX_WEIGHT = 0.2
AGE_SCALE = 95

np.random.seed(RANDOM_STATE)

GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]

DATA_DIR = next(
    (p for p in [Path("dataset"), Path("../dataset")] if (p / "u.data.csv").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("dataset/ introuvable")

notes = pd.read_csv(DATA_DIR / "u.data.csv")
films = pd.read_csv(DATA_DIR / "u.item.csv")
users = pd.read_csv(DATA_DIR / "u.user.csv", dtype={"zip_code": "string"}).drop(columns=["occupation"])

df_raw = notes.merge(users, on="user_id", how="left").merge(
    films, left_on="item_id", right_on="movie_id", how="left"
)
print(f"Genre unknown exclu : {int((df_raw['unknown']==1).sum())} notes")

df = df_raw.loc[df_raw["unknown"] != 1].drop(columns=["unknown"]).copy()
df["genres"] = df[GENRES].apply(lambda row: [g for g in GENRES if row[g] == 1], axis=1)

print(f"df : {len(df):,} notes | {df['user_id'].nunique()} users | {df['item_id'].nunique()} films")
display(df[["user_id", "item_id", "rating", "age", "gender", "title", "genres"]].head(3))


Genre unknown exclu : 10 notes
df : 99,990 notes | 943 users | 1680 films


,user_id,item_id,rating,age,gender,title,genres
0,196,242,3,49,M,Kolya (1996),[Comedy]
1,186,302,3,39,F,L.A. Confidential (1997),"[Crime, Film-Noir, Mystery, Thriller]"
2,22,377,1,25,M,Heavyweights (1994),"[Children's, Comedy]"


## 2. Split sans fuite

Split aleatoire. Stats et fit modeles **uniquement sur le train**.


In [126]:
train = df.sample(frac=TRAIN_FRAC, random_state=RANDOM_STATE)
test = df.drop(train.index)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f"Train {len(train):,} | Test {len(test):,}")

movies = df.drop_duplicates("item_id").set_index("item_id")
movie_genres = movies["genres"].to_dict()
movie_title = movies["title"].to_dict()
film_genre_X = movies[GENRES].astype(float)
all_movie_ids = film_genre_X.index.to_numpy()
print(f"Films catalogue : {len(all_movie_ids)}")


Train 24,998 | Test 74,992
Films catalogue : 1680


## 3. Stats train

Note moyenne et volume de notes par film (baseline).


In [127]:
film_stats = (
    train.groupby("item_id")["rating"].agg(n_ratings="count", mean_rating="mean").reset_index()
)
eligible = film_stats.loc[film_stats["n_ratings"] >= MIN_SUPPORT].set_index("item_id")
mean_rating = eligible["mean_rating"]
print(f"Films eligible (>= {MIN_SUPPORT} notes) : {len(eligible)}")
display(eligible.head(3))


Films eligible (>= 19 notes) : 457


,n_ratings,mean_rating
item_id,,
1,109,3.871560
2,30,3.200000
3,19,3.263158


## 4. Cas d'evaluation (1 à 3 films en entrée)

Requete = 1 à 3 films aimes en train ; pertinents = aimes en test.


In [128]:
liked_train = train.loc[train["rating"] >= LIKED_THRESHOLD]
liked_test = test.loc[test["rating"] >= LIKED_THRESHOLD]
train_counts = train.groupby("item_id").size()
cand_A = liked_train.copy()
cand_A["pop"] = cand_A["item_id"].map(train_counts)

# Extraction de 1 à 3 films aimés en train par utilisateur
def _get_top_seeds(df_u, max_n=MAX_SEEDS):
    sorted_u = df_u.sort_values(["rating", "pop"], ascending=[False, False])
    return list(sorted_u["item_id"].unique()[:max_n])

seeds_per_user = cand_A.groupby("user_id", group_keys=False).apply(_get_top_seeds).rename("query_ids")
user_demo = train.drop_duplicates("user_id").set_index("user_id")[["age", "gender"]]
relevant = liked_test.groupby("user_id")["item_id"].apply(set).rename("relevant_ids")
liked_lists = liked_train.groupby("user_id")["item_id"].apply(set)

eval_cases = pd.concat([seeds_per_user, user_demo, relevant], axis=1, join="inner").reset_index()
eval_cases["liked_train_ids"] = eval_cases["user_id"].map(liked_lists)
eval_cases["relevant_ids"] = eval_cases.apply(
    lambda r: r["relevant_ids"] - set(r["query_ids"]),
    axis=1,
)
eval_cases = eval_cases.loc[eval_cases["relevant_ids"].map(len) > 0].reset_index(drop=True)

print(f"Cas d'evaluation (1-{MAX_SEEDS} films) : {len(eval_cases)}")
print(f"Nombre moyen de films en entrée : {eval_cases['query_ids'].map(len).mean():.2f}")
print(f"Taille moyenne pertinents : {eval_cases['relevant_ids'].map(len).mean():.1f}")
display(eval_cases.head(3))


Cas d'evaluation (1-3 films) : 934
Nombre moyen de films en entrée : 2.88
Taille moyenne pertinents : 44.4


,user_id,query_ids,age,gender,relevant_ids,liked_train_ids
0,1,"[50, 174, 89]",24,M,"{1, 3, 9, 12, 14, 15, 16, 18, 19, 22, 23, 28, ...","{134, 7, 6, 265, 13, 20, 152, 25, 162, 166, 16..."
1,2,"[302, 311, 286]",53,F,"{1, 13, 14, 269, 272, 273, 275, 277, 279, 25, ...","{257, 297, 301, 302, 303, 111, 304, 276, 311, ..."
2,3,"[181, 329]",23,M,"{320, 321, 260, 327, 328, 331, 340, 342, 344, ...","{329, 181}"


## 5. Entrainement des modeles (train uniquement)

### Film–Film CF avec TF-IDF
Pondération TF-IDF des interactions.

### User–User CF (protocole produit)
Voisins = users train ayant aimé les films d'entrée, pondérés par age/sexe.

### Hybride Film×User + Genre Boost
`HYBRID_ALPHA * FilmCF + (1 - HYBRID_ALPHA) * UserCF`.


In [129]:
user_profile = (
    train.drop_duplicates("user_id")
    .set_index("user_id")[["age", "gender"]]
    .copy()
)

users_u = sorted(train["user_id"].unique())
items_u = sorted(train["item_id"].unique())
u_ix = {u: i for i, u in enumerate(users_u)}
i_ix = {it: i for i, it in enumerate(items_u)}

mat_ui = csr_matrix(
    (
        train["rating"].astype(float).values,
        (train["user_id"].map(u_ix).values, train["item_id"].map(i_ix).values),
    ),
    shape=(len(users_u), len(items_u)),
)
mat_iu = mat_ui.T.tocsr()

# Film–Film CF avec TF-IDF
tfidf = TfidfTransformer()
mat_iu_tfidf = tfidf.fit_transform(mat_iu)

nn_item = NearestNeighbors(metric="cosine", algorithm="brute")
nn_item.fit(mat_iu_tfidf)
item_ids_arr = np.array(items_u)

CF_ITEM = {}
n_ask_i = min(CF_N_FILM + 1, len(items_u))
d_i, ix_i = nn_item.kneighbors(mat_iu_tfidf, n_neighbors=n_ask_i)
for i, mid in enumerate(items_u):
    CF_ITEM[mid] = [
        (int(item_ids_arr[j]), 1.0 - float(d))
        for d, j in zip(d_i[i], ix_i[i])
        if int(item_ids_arr[j]) != mid
    ]

# User-CF
liked_by_user = (
    train.loc[train["rating"] >= LIKED_THRESHOLD]
    .groupby("user_id")[["item_id", "rating"]]
    .apply(lambda x: list(zip(x["item_id"], x["rating"])))
    .to_dict()
)

fans_of = {
    mid: list(
        zip(
            grp["user_id"].to_numpy(),
            grp["rating"].to_numpy(),
            grp["age"].to_numpy(),
            grp["gender"].to_numpy(),
        )
    )
    for mid, grp in train.loc[train["rating"] >= LIKED_THRESHOLD].groupby("item_id")
}

print(f"Film–Film CF (TF-IDF) : {len(CF_ITEM)} films")
print(f"User–User CF : voisins = fans(seeds) x age/sexe")
print(f"Hybride Film x User : alpha_film={HYBRID_ALPHA}")


Film–Film CF (TF-IDF) : 1450 films
User–User CF : voisins = fans(seeds) x age/sexe
Hybride Film x User : alpha_film=0.05


## 6. Recommenders (support multi-films)

Exclure les films requetes et les films deja aimes en train.


In [130]:
def _seen(query_ids, liked_train_ids):
    if isinstance(query_ids, (int, np.integer)):
        seeds = {int(query_ids)}
    else:
        seeds = set(int(x) for x in query_ids)
    s = set(liked_train_ids) if liked_train_ids is not None else set()
    return s.union(seeds), seeds


def recommend_dummy(query_ids, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen, seeds = _seen(query_ids, liked_train_ids)
    pool = np.array([m for m in all_movie_ids if m not in seen])
    seed_val = sum(seeds)
    local = np.random.default_rng(RANDOM_STATE + seed_val)
    if len(pool) == 0:
        return []
    return local.choice(pool, size=min(k, len(pool)), replace=False).tolist()


def recommend_baseline(query_ids, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen, seeds = _seen(query_ids, liked_train_ids)
    union_g = set()
    for qid in seeds:
        union_g.update(movie_genres.get(qid, []))
    if not union_g:
        return []
    scores = [
        (mid, float(m))
        for mid, m in mean_rating.items()
        if mid not in seen and union_g & set(movie_genres.get(mid, []))
    ]
    scores.sort(key=lambda x: (-x[1], x[0]))
    return [m for m, _ in scores[:k]]


def _demo_sim(age_a, gender_a, age_b, gender_b):
    sex = 1.0 if gender_a == gender_b else 0.0
    age_term = 1.0 - min(abs(float(age_a) - float(age_b)) / AGE_SCALE, 1.0)
    return DEMO_SEX_WEIGHT * sex + (1.0 - DEMO_SEX_WEIGHT) * age_term


def recommend_user_cf(query_ids, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen, seeds = _seen(query_ids, liked_train_ids)
    age = 30.0 if age is None else float(age)
    gender = "M" if gender is None else gender

    neighbors = []
    for qid in seeds:
        for uid, rat, a, g in fans_of.get(qid, []):
            demo = _demo_sim(age, gender, a, g)
            neighbors.append((int(uid), demo * float(rat) / 5.0))
    neighbors.sort(key=lambda x: -x[1])
    neighbors = neighbors[:CF_N_USER]

    scores = {}
    for vid, sim in neighbors:
        for mid, rat in liked_by_user.get(vid, []):
            if mid in seen:
                continue
            scores[mid] = scores.get(mid, 0.0) + float(sim) * float(rat)

    return [m for m, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]]


def recommend_film_cf(query_ids, k=10, age=None, gender=None, liked_train_ids=None, user_id=None):
    seen, seeds = _seen(query_ids, liked_train_ids)
    all_seeds = set(seeds)
    if liked_train_ids is not None:
        all_seeds.update(liked_train_ids)

    seed_ratings = {}
    if user_id is not None:
        sub = train.loc[
            (train["user_id"] == user_id) & (train["rating"] >= LIKED_THRESHOLD),
            ["item_id", "rating"],
        ]
        seed_ratings = dict(zip(sub["item_id"], sub["rating"]))

    scores = {}
    for aid in all_seeds:
        w = float(seed_ratings.get(aid, LIKED_THRESHOLD))
        for mid, sim in CF_ITEM.get(aid, []):
            if mid in seen:
                continue
            scores[mid] = scores.get(mid, 0.0) + w * sim
    return [m for m, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]]


def recommend_hybrid(query_ids, k=10, age=None, gender=None, liked_train_ids=None, user_id=None,
                      alpha=None, pool=HYBRID_POOL):
    alpha = HYBRID_ALPHA if alpha is None else float(alpha)
    pool = max(pool, k)
    film_list = recommend_film_cf(
        query_ids, k=pool, age=age, gender=gender,
        liked_train_ids=liked_train_ids, user_id=user_id,
    )
    user_list = recommend_user_cf(
        query_ids, k=pool, age=age, gender=gender,
        liked_train_ids=liked_train_ids, user_id=user_id,
    )
    scores = {}
    for rank, mid in enumerate(film_list):
        scores[mid] = scores.get(mid, 0.0) + alpha / (1.0 + rank)
    for rank, mid in enumerate(user_list):
        scores[mid] = scores.get(mid, 0.0) + (1.0 - alpha) / (1.0 + rank)

    _, seeds = _seen(query_ids, liked_train_ids)
    union_g = set()
    for qid in seeds:
        union_g.update(movie_genres.get(qid, []))

    for mid in list(scores.keys()):
        cand_g = set(movie_genres.get(mid, []))
        if union_g & cand_g:
            scores[mid] *= 1.15

    return [m for m, _ in sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:k]]


ex = eval_cases.iloc[0]
qids = ex["query_ids"]
kw = dict(
    age=ex["age"], gender=ex["gender"],
    liked_train_ids=ex["liked_train_ids"], user_id=int(ex["user_id"]),
)
print("Query seeds :", [movie_title.get(i) for i in qids])
print("Dummy   :", [movie_title.get(i) for i in recommend_dummy(qids, 5, **kw)])
print("Baseline:", [movie_title.get(i) for i in recommend_baseline(qids, 5, **kw)])
print("UserCF  :", [movie_title.get(i) for i in recommend_user_cf(qids, 5, **kw)])
print("FilmCF  :", [movie_title.get(i) for i in recommend_film_cf(qids, 5, **kw)])
print("Hybrid  :", [movie_title.get(i) for i in recommend_hybrid(qids, 5, **kw)])


Query seeds : ['Star Wars (1977)', 'Raiders of the Lost Ark (1981)', 'Blade Runner (1982)']
Dummy   : ['Shadow, The (1994)', 'Fled (1996)', 'Paris Was a Woman (1995)', 'Beautiful Girls (1996)', 'Fear of a Black Hat (1993)']
Baseline: ['Maltese Falcon, The (1941)', "Schindler's List (1993)", 'Casablanca (1942)', 'Manchurian Candidate, The (1962)', 'Bridge on the River Kwai, The (1957)']
UserCF  : ['Return of the Jedi (1983)', 'Godfather, The (1972)', 'Monty Python and the Holy Grail (1974)', 'Pulp Fiction (1994)', 'Empire Strikes Back, The (1980)']
FilmCF  : ['Monty Python and the Holy Grail (1974)', 'Empire Strikes Back, The (1980)', 'Princess Bride, The (1987)', 'Return of the Jedi (1983)', 'Unforgiven (1992)']
Hybrid  : ['Return of the Jedi (1983)', 'Godfather, The (1972)', 'Monty Python and the Holy Grail (1974)', 'Empire Strikes Back, The (1980)', 'Pulp Fiction (1994)']


## 7. Evaluation Precision@5 et @10 (avec historique train)


In [131]:
def precision_at_k(recommended, relevant, k):
    top = recommended[:k]
    if not top:
        return 0.0
    return sum(1 for m in top if m in relevant) / k


def evaluate(recommend_fn, cases, ks=(5, 10), max_k=10):
    rows, cache = [], []
    for _, row in cases.iterrows():
        qids = row["query_ids"]
        relevant = row["relevant_ids"]
        rec = recommend_fn(
            qids,
            k=max_k,
            age=row["age"],
            gender=row["gender"],
            liked_train_ids=row["liked_train_ids"],
            user_id=int(row["user_id"]),
        )
        cache.append({"user_id": row["user_id"], "query_ids": qids, "rec": rec, "relevant": relevant})
        rows.append({f"P@{k}": precision_at_k(rec, relevant, k) for k in ks})
    return pd.DataFrame(rows).mean().to_frame("mean").T, cache


methods = [
    ("dummy_random", recommend_dummy),
    ("baseline_overlap_mean", recommend_baseline),
    ("model_user_user_cf", recommend_user_cf),
    ("model_film_film_cf", recommend_film_cf),
    ("model_hybrid_film_user", recommend_hybrid),
]

summaries, caches = [], {}
for name, fn in methods:
    s, c = evaluate(fn, eval_cases)
    summaries.append(s.assign(method=name))
    caches[name] = c

results = pd.concat(summaries, ignore_index=True)[["method", "P@5", "P@10"]]
print(f"Precision moyenne sur {len(eval_cases)} cas (seed={RANDOM_STATE})")
display(results.round(4))


Precision moyenne sur 934 cas (seed=42)


,method,P@5,P@10
0,dummy_random,0.0321,0.0285
1,baseline_overlap_mean,0.2049,0.1849
2,model_user_user_cf,0.3979,0.3495
3,model_film_film_cf,0.2443,0.2436
4,model_hybrid_film_user,0.3976,0.3528


## 8. Test situation produit (age + sexe + 1 à 3 films d'entrée)

Meme metriques, mais **sans** `user_id` MovieLens (pure entrée cold-start utilisateur).


In [132]:
def evaluate_product(recommend_fn, cases, ks=(5, 10), max_k=10):
    rows = []
    for _, row in cases.iterrows():
        qids = row["query_ids"]
        relevant = row["relevant_ids"]
        rec = recommend_fn(
            qids,
            k=max_k,
            age=row["age"],
            gender=row["gender"],
            liked_train_ids=None,
            user_id=None,
        )
        rows.append({f"P@{k}": precision_at_k(rec, relevant, k) for k in ks})
    return pd.DataFrame(rows).mean().to_frame("mean").T

summaries_prod = []
for name, fn in methods:
    s = evaluate_product(fn, eval_cases)
    summaries_prod.append(s.assign(method=name))

results_product = pd.concat(summaries_prod, ignore_index=True)[["method", "P@5", "P@10"]]
print(
    f"Situation produit (âge + sexe + 1 à 3 films) — Precision moyenne "
    f"sur {len(eval_cases)} cas (seed={RANDOM_STATE})"
)
display(results_product.round(4))

print("\nRappel section 7 (éval avec historique train, CF multi-seed) :")
display(results.round(4))


Situation produit (âge + sexe + 1 à 3 films) — Precision moyenne sur 934 cas (seed=42)


,method,P@5,P@10
0,dummy_random,0.0268,0.0261
1,baseline_overlap_mean,0.1878,0.1759
2,model_user_user_cf,0.3565,0.3105
3,model_film_film_cf,0.2317,0.2211
4,model_hybrid_film_user,0.3567,0.3101



Rappel section 7 (éval avec historique train, CF multi-seed) :


,method,P@5,P@10
0,dummy_random,0.0321,0.0285
1,baseline_overlap_mean,0.2049,0.1849
2,model_user_user_cf,0.3979,0.3495
3,model_film_film_cf,0.2443,0.2436
4,model_hybrid_film_user,0.3976,0.3528
